In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from scipy.sparse import hstack

np.random.seed(7)




In [2]:
possible_dirs = [
    os.path.join("input", "spooky-author-identification"),
    os.path.join("/kaggle", "input", "spooky-author-identification"),
    os.path.join("data", "spooky-author-identification"),
    os.path.join("/kaggle", "working", "spooky-author-identification"),
    "spooky-author-identification",
]
base_dir = None
for d in possible_dirs:
    if os.path.isdir(d):
        base_dir = d
        break
if base_dir is None:
    raise FileNotFoundError(
        "Data directory for spooky-author-identification not found."
    )
train_path = os.path.join(base_dir, "train.csv")
test_path = os.path.join(base_dir, "test.csv")
sample_sub_path = os.path.join(base_dir, "sample_submission.csv")




In [3]:
df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_full = pd.concat([df, df_test], sort=False)

a2c = {"EAP": 0, "HPL": 1, "MWS": 2}
y_int = np.array([a2c[a] for a in df.author])




In [4]:
counter = {name: defaultdict(int) for name in set(df.author)}
for text, author in zip(df.text, df.author):
    text = text.replace(" ", "")
    for c in text:
        counter[author][c] += 1

chars = set()
for v in counter.values():
    chars |= v.keys()

names = [author for author in counter.keys()]

print("c ", end="")
for n in names:
    print(n, end="   ")
print()
for c in chars:
    print(c, end=" ")
    for n in names:
        print(counter[n][c], end=" ")
    print()




c HPL   EAP   MWS   
R 209 233 348 
e 79256 102699 88221 
Z 43 15 2 
Q 10 20 7 
' 1573 1176 426 
ñ 7 0 0 
. 5332 7573 5206 
S 757 651 521 
E 247 391 408 
U 80 151 43 
; 1028 1211 2393 
, 7717 15651 10863 
s 39466 48003 41589 
ë 10 0 0 
ö 2 12 0 
à 0 8 0 
F 236 337 219 
A 1041 1120 847 
B 470 755 355 
z 475 566 359 
ü 5 1 0 
V 61 140 51 
m 15879 20378 18513 
f 14538 19981 16537 
t 55884 73704 57079 
H 657 791 606 
Υ 1 0 0 
C 390 372 275 
ê 2 26 0 
ç 0 1 0 
v 5843 8571 7165 
é 15 43 0 
: 44 163 299 
c 16502 21605 16196 
i 39750 54488 41549 
" 461 2678 1337 
a 50984 61285 49813 
K 156 75 30 
J 193 145 60 
è 0 13 0 
ä 3 1 0 
o 45784 60172 48209 
x 930 1766 1139 
Y 94 247 212 
q 681 918 610 
D 300 443 208 
X 5 14 3 
Å 1 0 0 
Ν 1 0 0 
L 232 411 280 
I 3140 4339 4464 
î 0 1 0 
W 649 671 607 
P 287 382 314 
T 1423 1986 1121 
M 580 971 370 
Π 1 0 0 
h 38489 46016 39427 
b 9585 11851 8664 
u 17463 23540 18966 
æ 10 32 0 
d 29982 32981 31906 
Σ 1 0 0 
â 0 6 0 
l 27162 31655 25074 
ô 0 7 0 
k 4746

In [5]:
def preprocess(text):
    text = text.replace("' ", " ' ")
    signs = set(',.:;"?!')
    prods = set(text) & signs
    if not prods:
        return text
    for sign in prods:
        text = text.replace(sign, f" {sign} ")
    return text


def create_docs(df, n_gram_max=3):
    def add_ngram(q, n_gram_max):
        ngrams = []
        for n in range(2, n_gram_max + 1):
            for w_index in range(len(q) - n + 1):
                ngrams.append("--".join(q[w_index : w_index + n]))
        return q + ngrams

    docs = []
    for doc in df.text:
        doc = preprocess(doc).split()
        docs.append(" ".join(add_ngram(doc, n_gram_max)))
    return docs




In [6]:
train_texts_raw, val_texts_raw, y_train, y_val = train_test_split(
    df.text.astype(str).values,
    y_int,
    test_size=0.20,
    random_state=42,
    stratify=y_int,
)

train_texts = create_docs(pd.DataFrame({"text": train_texts_raw}), n_gram_max=3)
val_texts = create_docs(pd.DataFrame({"text": val_texts_raw}), n_gram_max=3)

# Expanded TF‑IDF: keep all n‑grams (min_df=1) and extend character n‑gram length to 6
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 6),
    min_df=1,
    sublinear_tf=True,
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

X_char_train = char_vectorizer.fit_transform(train_texts)
X_word_train = word_vectorizer.fit_transform(train_texts)
X_train = hstack([X_char_train, X_word_train])

X_char_val = char_vectorizer.transform(val_texts)
X_word_val = word_vectorizer.transform(val_texts)
X_val = hstack([X_char_val, X_word_val])




In [7]:
clf = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    n_jobs=5,
    random_state=7,
    C=8.0,  # less regularisation (more capacity) to improve fit
    class_weight="balanced",
)

clf.fit(X_train, y_train)




LogisticRegression(C=8.0, class_weight='balanced', max_iter=1000,
                   multi_class='multinomial', n_jobs=5, random_state=7)

In [8]:
val_pred = clf.predict_proba(X_val)
val_loss = log_loss(y_val, val_pred)
print(f"Validation log‑loss: {val_loss:.5f}")




Validation log‑loss: 0.35939


In [9]:
test_df = pd.read_csv(test_path)
test_texts = test_df.text.astype(str).values

X_char_test = char_vectorizer.transform(test_texts)
X_word_test = word_vectorizer.transform(test_texts)
X_test = hstack([X_char_test, X_word_test])

y_pred = clf.predict_proba(X_test)  # shape (n_test, 3)




In [10]:
result = pd.read_csv(sample_sub_path)
for author, idx in a2c.items():
    result[author] = y_pred[:, idx]

submission_path = "fastText_result_05.csv"
result.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to fastText_result_05.csv
